# Fine-tuning de BERT para Classificação de Mensagens de Clientes

**Objetivo:** Treinar um modelo BERT para classificar mensagens de clientes de uma rede varejista em duas categorias:
- **venda** — intenção de compra
- **suporte** — dúvidas ou problemas com produtos

**Modelo base:** `neuralmind/bert-base-portuguese-cased` (BERTimbau — BERT pré-treinado em português)

## 1. Instalação de Dependências

In [ ]:
!pip install transformers datasets torch scikit-learn accelerate -q

## 2. Imports

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 3. Carregamento e Exploração dos Dados

In [ ]:
def load_jsonl(path: str) -> list[dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

train_raw = load_jsonl("treino.jsonl")
test_raw = load_jsonl("teste.jsonl")

print(f"Exemplos de treino: {len(train_raw)}")
print(f"Exemplos de teste:  {len(test_raw)}")
print(f"\nAmostra de treino:")
for ex in train_raw[:3]:
    print(f"  [{ex['completion']}] {ex['prompt']}")

In [ ]:
from collections import Counter

train_labels = [ex["completion"] for ex in train_raw]
test_labels = [ex["completion"] for ex in test_raw]

print("Distribuição no treino:", Counter(train_labels))
print("Distribuição no teste: ", Counter(test_labels))

## 4. Preparação do Dataset

Mapeamento de labels para inteiros e criação dos objetos `Dataset` do Hugging Face.

In [ ]:
LABEL2ID = {"venda": 0, "suporte": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def build_dataset(raw_data: list[dict]) -> Dataset:
    return Dataset.from_dict({
        "text": [ex["prompt"] for ex in raw_data],
        "label": [LABEL2ID[ex["completion"]] for ex in raw_data],
    })

train_dataset = build_dataset(train_raw)
test_dataset = build_dataset(test_raw)

datasets = DatasetDict({"train": train_dataset, "test": test_dataset})
print(datasets)

## 5. Tokenização com BERTimbau

In [ ]:
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )

tokenized = datasets.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized.set_format("torch")

print("Colunas após tokenização:", tokenized["train"].column_names)
print("Shape input_ids:", tokenized["train"][0]["input_ids"].shape)

## 6. Split de Validação

Separamos 20% do treino para validação durante o treinamento.

In [ ]:
split = tokenized["train"].train_test_split(test_size=0.2, seed=42, stratify_by_column="label")

train_split = split["train"]
val_split = split["test"]
test_split = tokenized["test"]

print(f"Treino:    {len(train_split)} exemplos")
print(f"Validação: {len(val_split)} exemplos")
print(f"Teste:     {len(test_split)} exemplos")

## 7. Carregamento do Modelo

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total de parâmetros:     {total_params:,}")
print(f"Parâmetros treináveis:   {trainable_params:,}")

## 8. Métricas de Avaliação

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

## 9. Configuração do Treinamento

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=10,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=val_split,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer configurado com sucesso!")

## 10. Treinamento (Fine-tuning)

In [ ]:
train_result = trainer.train()

print("\n=== Resultados do Treinamento ===")
print(f"  Tempo total: {train_result.metrics['train_runtime']:.1f}s")
print(f"  Loss final:  {train_result.metrics['train_loss']:.4f}")

## 11. Avaliação no Conjunto de Teste

In [ ]:
eval_results = trainer.evaluate(test_split)

print("=== Métricas no Conjunto de Teste ===")
print(f"  Accuracy:  {eval_results['eval_accuracy']:.4f}")
print(f"  F1-Score:  {eval_results['eval_f1']:.4f}")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall:    {eval_results['eval_recall']:.4f}")

## 12. Análise Detalhada — Matriz de Confusão e Classification Report

In [ ]:
predictions = trainer.predict(test_split)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print("=== Classification Report ===")
print(classification_report(
    true_labels, preds,
    target_names=["venda", "suporte"],
))

cm = confusion_matrix(true_labels, preds)
print("=== Matriz de Confusão ===")
print(f"              Pred: venda  Pred: suporte")
print(f"  Real: venda    {cm[0][0]:>5}        {cm[0][1]:>5}")
print(f"  Real: suporte  {cm[1][0]:>5}        {cm[1][1]:>5}")

## 13. Salvar o Modelo Fine-tuned

In [ ]:
SAVE_DIR = "./modelo-classificador-atendimento"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Modelo salvo em: {SAVE_DIR}")

## 14. Inferência — Testando o Modelo com Novas Mensagens

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

mensagens_novas = [
    "Quero saber o preço do ventilador de teto.",
    "Meu pedido chegou errado, o que faço?",
    "Vocês têm forno elétrico à venda?",
    "O controle remoto da TV parou de funcionar.",
    "Quanto custa o jogo de panelas inox?",
    "Não consigo ligar o purificador de água.",
]

print("=== Inferência em Mensagens Novas ===\n")
resultados = classifier(mensagens_novas)

for msg, res in zip(mensagens_novas, resultados):
    label = res["label"]
    score = res["score"]
    print(f"  [{label:>7}] ({score:.2%}) \"{msg}\"")

## 15. Conclusão

### O que foi feito:
1. **Dados carregados** dos arquivos JSONL (treino e teste)
2. **Tokenização** com BERTimbau (`neuralmind/bert-base-portuguese-cased`)
3. **Fine-tuning** com `Trainer` do Hugging Face (5 épocas, early stopping)
4. **Avaliação** com accuracy, F1, precision, recall e matriz de confusão
5. **Modelo salvo** para uso em produção via `pipeline`

### Por que BERTimbau?
- BERT pré-treinado **especificamente em português** brasileiro
- Excelente para tarefas de classificação de texto em PT-BR
- Modelo leve (~110M parâmetros) e eficiente para fine-tuning